# TabuLLM: Advanced Pipelines

This notebook demonstrates how TabuLLM's building blocks compose into more sophisticated ML pipelines,
continuing from `01_fraud_detection_walkthrough.ipynb`.

Two patterns are explored on the same fraud detection dataset:

1. **Column concatenation sweep** — progressively embedding 1–7 text columns to measure marginal
   contribution and identify diminishing returns
2. **Stacking ensembles** — processing column groups independently and combining predictions via a
   meta-learner, extracting more signal than monolithic concatenation

**Requirements:**
- `pip install tabullm[examples]`
- Set `EMBEDDING_PROVIDER` in the configuration cell below.
- `"hf"` runs locally (free, no API key needed; slower when embedding many column subsets).
- `"bedrock"` and `"openai"` use API calls (faster; require credentials in `.env`).
- All embeddings are cached in memory: each unique column subset is embedded exactly once per session.

In [1]:
# --- Configuration ---
EMBEDDING_PROVIDER  = "bedrock"  # "hf", "bedrock", or "openai"
RANDOM_STATE        = 42
N_TREES_SWEEP       = 100        # Trees per pipeline in the column sweep (Section 2)
N_TREES_COMPARISON  = 1000       # Total trees for the final comparison table (Section 4)
                                  # Set to 100 for a fast run; 1000 reproduces the paper results

In [2]:
import time
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, TargetEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

from tabullm import TextColumnTransformer, GMMFeatureExtractor, load_fraud

load_dotenv(override=True)

/Users/amahani/code/TabuLLM/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

## 1. Setup

In [3]:
X, y, metadata = load_fraud()

text_cols    = metadata['text_columns']
binary_cols  = metadata['binary_columns']
low_card_cols = ['employment_type', 'required_experience', 'required_education']
med_card_cols = ['industry', 'function']

# Reset index so integer positions and pandas index stay aligned throughout
X = X.reset_index(drop=True)
y = y.reset_index(drop=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=RANDOM_STATE
)

print(f"Train: {len(X_train):,}  |  Test: {len(X_test):,}")
print(f"Text columns: {text_cols}")

Train: 12,516  |  Test: 5,364
Text columns: ['title', 'location', 'department', 'company_profile', 'description', 'requirements', 'benefits']


In [4]:
# Initialize embedding model
if EMBEDDING_PROVIDER == "hf":
    from langchain_huggingface import HuggingFaceEmbeddings
    embedding_model = HuggingFaceEmbeddings(
        model_name='sentence-transformers/all-MiniLM-L6-v2'
    )
elif EMBEDDING_PROVIDER == "bedrock":
    from langchain_aws import BedrockEmbeddings
    embedding_model = BedrockEmbeddings(
        model_id='amazon.titan-embed-text-v2:0',
        region_name='us-east-1'
    )
elif EMBEDDING_PROVIDER == "openai":
    from langchain_openai import OpenAIEmbeddings
    embedding_model = OpenAIEmbeddings(model='text-embedding-3-small')
else:
    raise ValueError(f"Unknown EMBEDDING_PROVIDER: {EMBEDDING_PROVIDER!r}")

# normalize=True ensures L2-unit vectors regardless of provider
text_transformer = TextColumnTransformer(model=embedding_model, normalize=True)

print(f"Embedding provider: {EMBEDDING_PROVIDER}")

Embedding provider: bedrock


In [5]:
# ── Session-level embedding cache ────────────────────────────────────────────
# Each unique column subset is embedded exactly once per session.
# Subsequent calls with the same name return the cached array instantly.
_emb_cache = {}

def embed_cols(cols, name):
    """Embed a column subset, caching the result by name.
    
    Returns (array, col_names) where array is (n_samples, emb_dim)
    and col_names is a list of DataFrame column names for this embedding.
    """
    if name not in _emb_cache:
        print(f"  Embedding '{name}' ({list(cols)}) ... ", end='', flush=True)
        t0 = time.time()
        arr = np.asarray(text_transformer.fit_transform(X[list(cols)]))
        _emb_cache[name] = arr
        print(f"done  ({time.time() - t0:.1f}s, shape={arr.shape})")
    arr = _emb_cache[name]
    col_names = [f'emb_{name}_{i}' for i in range(arr.shape[1])]
    return arr, col_names


# ── Shared helpers ────────────────────────────────────────────────────────────

def nontext_transformers():
    """ColumnTransformer entries for structured (non-text) features."""
    return [
        ('binary', 'passthrough', binary_cols),
        ('low_card', Pipeline([
            ('impute', SimpleImputer(strategy='constant', fill_value='Missing')),
            ('onehot', OneHotEncoder(sparse_output=False, handle_unknown='ignore'))
        ]), low_card_cols),
        ('med_card', Pipeline([
            ('impute', SimpleImputer(strategy='constant', fill_value='Missing')),
            ('target', TargetEncoder(cv=5, smooth='auto', target_type='binary',
                                     random_state=RANDOM_STATE))
        ]), med_card_cols),
    ]


def make_base_estimator(emb_col_names, n_estimators, random_state=RANDOM_STATE):
    """Pipeline: pre-embedded columns → GMM → RF, combined with non-text features.
    
    Takes a list of column names corresponding to a pre-embedded array
    already present in the augmented DataFrame.
    """
    return Pipeline([
        ('features', ColumnTransformer([
            ('embeddings', GMMFeatureExtractor(
                n_components=10, n_init=3,
                include_log_density=True,
                random_state=random_state
            ), emb_col_names),
        ] + nontext_transformers())),
        ('rf', RandomForestClassifier(
            n_estimators=n_estimators,
            random_state=random_state
        ))
    ])

## 2. Column Concatenation Sweep

The fraud dataset has seven text columns varying in information content. A natural question:
do all columns contribute equally, or do returns diminish as more are added?

We answer this with two sweeps:
- **Forward**: add columns in standard order (title → benefits)
- **Backward**: add columns in reverse order (benefits → title)

Both sweeps use the same pipeline — embed the current subset, reduce via GMM, combine with
structured features, classify with a random forest. If both sweeps show the same pattern,
it is a property of the columns' information content, not their order.

**Note on pre-embedding:** `TextColumnTransformer` is called once per unique column subset
via the session cache. The pipelines in this section receive pre-computed embedding arrays
as named DataFrame columns, so `GMMFeatureExtractor` is the first transformer in each pipeline.
This is equivalent to having `TextColumnTransformer` inside the pipeline — since embedding is
row-wise and deterministic, the output is identical — but avoids redundant computation.

In [ ]:
text_cols_fwd = text_cols                        # title → benefits
text_cols_bwd = list(reversed(text_cols))        # benefits → title

# ── Non-text baseline ────────────────────────────────────────────────────────
baseline_pipe = Pipeline([
    ('features', ColumnTransformer(nontext_transformers())),
    ('rf', RandomForestClassifier(n_estimators=N_TREES_SWEEP, random_state=RANDOM_STATE))
])
baseline_pipe.fit(X_train, y_train)
baseline_auc = roc_auc_score(y_test, baseline_pipe.predict_proba(X_test)[:, 1])
print(f"Non-text baseline AUC: {baseline_auc:.4f}")
print()

# ── Forward sweep ────────────────────────────────────────────────────────────
print("Forward sweep (title → benefits):")
results_fwd = {}

for n in range(1, 8):
    cols = text_cols_fwd[:n]
    emb, emb_col_names = embed_cols(cols, f'fwd_{n}')

    emb_df = pd.DataFrame(emb, columns=emb_col_names)
    X_aug = pd.concat([X, emb_df], axis=1)

    pipe = make_base_estimator(emb_col_names, n_estimators=N_TREES_SWEEP)
    pipe.fit(X_aug.loc[X_train.index], y_train)
    auc = roc_auc_score(y_test, pipe.predict_proba(X_aug.loc[X_test.index])[:, 1])
    results_fwd[n] = auc
    print(f"  {n} col{'s' if n > 1 else ''}: AUC = {auc:.4f}  (Δ = {auc - baseline_auc:+.4f})")

In [ ]:
# ── Backward sweep ───────────────────────────────────────────────────────────
print("Backward sweep (benefits → title):")
results_bwd = {}

for n in range(1, 8):
    cols = text_cols_bwd[:n]
    emb, emb_col_names = embed_cols(cols, f'bwd_{n}')

    emb_df = pd.DataFrame(emb, columns=emb_col_names)
    X_aug = pd.concat([X, emb_df], axis=1)

    pipe = make_base_estimator(emb_col_names, n_estimators=N_TREES_SWEEP)
    pipe.fit(X_aug.loc[X_train.index], y_train)
    auc = roc_auc_score(y_test, pipe.predict_proba(X_aug.loc[X_test.index])[:, 1])
    results_bwd[n] = auc
    print(f"  {n} col{'s' if n > 1 else ''}: AUC = {auc:.4f}  (Δ = {auc - baseline_auc:+.4f})")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

ns = list(range(1, 8))
ax.plot(ns, [results_fwd[n] for n in ns], 'o-', label='Forward (title → benefits)', color='steelblue')
ax.plot(ns, [results_bwd[n] for n in ns], 's--', label='Backward (benefits → title)', color='darkorange')
ax.axhline(baseline_auc, color='gray', linestyle=':', linewidth=1.5, label='Non-text baseline')

ax.set_xlabel('Number of text columns')
ax.set_ylabel('Test AUC')
ax.set_title('Marginal contribution of text columns')
ax.set_xticks(ns)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

Both sweeps show the same pattern: performance gains are concentrated in the first few columns,
with little further improvement beyond 3–4 columns. The full seven-column concatenation
performs similarly to the four-column subset. This is a property of the columns' information
content, not their order — the backward sweep converges to the same plateau from the other direction.

A natural interpretation: when all seven columns are concatenated into a single embedding,
the dominant signal from the most informative columns gets diluted by noisier ones. Processing
column groups separately before combining predictions may better preserve each group's signal.
That is the motivation for the stacking approach in the next section.

## 3. Stacking Ensembles

A stacking ensemble splits the seven text columns into groups, embeds each group independently,
trains a separate base classifier per group, and combines their predictions via a meta-learner
(logistic regression). The base classifiers all receive the same structured features.

We explore two variants:
- **Single-split**: one fixed split of 3 + 4 columns
- **Multi-split**: five random splits (10 base classifiers total), averaging over grouping choices

**Implementation note:** The pre-embedded arrays for each column group are appended as named
columns to the original DataFrame. `StackingClassifier` receives this combined DataFrame and
each base estimator's `ColumnTransformer` selects its own embedding columns by name. During
sklearn's internal cross-validation for out-of-fold meta-features, only GMM + RF are re-fit
per fold — no re-embedding occurs.

### 3a. Single-split stacking

A natural first split separates the job-level identity columns (title, location, department)
from the longer free-text columns (company profile, description, requirements, benefits).

In [ ]:
split_a_cols = ['title', 'location', 'department']
split_b_cols = ['company_profile', 'description', 'requirements', 'benefits']

# Embed each group (cached; if fwd_3 / fwd_7 are already cached from the sweep,
# these are fresh calls since the column subsets differ from the sweep subsets)
emb_a, emb_a_cols = embed_cols(split_a_cols, 'ss_a')
emb_b, emb_b_cols = embed_cols(split_b_cols, 'ss_b')

# Build combined DataFrame: original features + both embedding groups
X_ss = pd.concat([
    X,
    pd.DataFrame(emb_a, columns=emb_a_cols),
    pd.DataFrame(emb_b, columns=emb_b_cols),
], axis=1)

X_ss_train = X_ss.loc[X_train.index]
X_ss_test  = X_ss.loc[X_test.index]

n_trees_per_est = N_TREES_SWEEP // 2   # keep total trees = N_TREES_SWEEP for fair comparison

stacking_ss = StackingClassifier(
    estimators=[
        ('group_a', make_base_estimator(emb_a_cols, n_trees_per_est)),
        ('group_b', make_base_estimator(emb_b_cols, n_trees_per_est)),
    ],
    final_estimator=LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    cv=5
)

stacking_ss.fit(X_ss_train, y_train)
auc_ss = roc_auc_score(y_test, stacking_ss.predict_proba(X_ss_test)[:, 1])

print(f"Single-split stacking AUC: {auc_ss:.4f}")
print(f"vs. concatenated baseline:  {results_fwd[7]:.4f}  (Δ = {auc_ss - results_fwd[7]:+.4f})")

### 3b. Multi-split stacking

The single-split result depends on the particular column grouping chosen. To reduce sensitivity
to any one split and exploit additional diversity, we average over multiple random splits:
for each of `N_SPLITS` random permutations of the seven columns, we create two base estimators
(groups of 3 and 4), yielding `2 × N_SPLITS` base classifiers in total.

In [ ]:
N_SPLITS = 5
rng = np.random.default_rng(seed=RANDOM_STATE)

# Generate random column splits and embed each group
print(f"Generating {N_SPLITS} random splits and embedding each group:")
splits = []
for i in range(N_SPLITS):
    shuffled = rng.permutation(text_cols)
    group_a = sorted(shuffled[:3].tolist())   # sort for reproducible column order
    group_b = sorted(shuffled[3:].tolist())

    emb_a, emb_a_cols = embed_cols(group_a, f'ms_{i}a')
    emb_b, emb_b_cols = embed_cols(group_b, f'ms_{i}b')
    splits.append({
        'group_a': group_a, 'group_b': group_b,
        'emb_a': emb_a, 'emb_a_cols': emb_a_cols,
        'emb_b': emb_b, 'emb_b_cols': emb_b_cols,
    })
    print(f"  Split {i+1}: A={group_a}")
    print(f"            B={group_b}")

In [ ]:
# Build combined DataFrame: original features + all 10 embedding groups
dfs = [X]
for s in splits:
    dfs.append(pd.DataFrame(s['emb_a'], columns=s['emb_a_cols']))
    dfs.append(pd.DataFrame(s['emb_b'], columns=s['emb_b_cols']))
X_ms = pd.concat(dfs, axis=1)

X_ms_train = X_ms.loc[X_train.index]
X_ms_test  = X_ms.loc[X_test.index]

n_trees_per_est = N_TREES_SWEEP // (N_SPLITS * 2)  # keep total = N_TREES_SWEEP
n_trees_per_est = max(n_trees_per_est, 10)          # floor at 10

base_estimators = []
for i, s in enumerate(splits):
    base_estimators.append((f'split{i+1}a', make_base_estimator(s['emb_a_cols'], n_trees_per_est)))
    base_estimators.append((f'split{i+1}b', make_base_estimator(s['emb_b_cols'], n_trees_per_est)))

stacking_ms = StackingClassifier(
    estimators=base_estimators,
    final_estimator=LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    cv=5
)

stacking_ms.fit(X_ms_train, y_train)
auc_ms = roc_auc_score(y_test, stacking_ms.predict_proba(X_ms_test)[:, 1])

print(f"Multi-split stacking AUC ({N_SPLITS} splits, {len(base_estimators)} base estimators): {auc_ms:.4f}")
print(f"vs. single-split:           {auc_ss:.4f}  (Δ = {auc_ms - auc_ss:+.4f})")
print(f"vs. concatenated baseline:  {results_fwd[7]:.4f}  (Δ = {auc_ms - results_fwd[7]:+.4f})")

## 4. Comparison at Equal Modeling Capacity

The sweep and stacking runs above used `N_TREES_SWEEP` trees for quick illustration.
To fairly compare all four configurations, we re-run with `N_TREES_COMPARISON` total trees,
allocated proportionally across base estimators. All embeddings are already cached.

Expected runtimes at `N_TREES_COMPARISON = 1000` on a modern laptop:
- Non-text baseline and single model: ~15–30s each
- Single-split stacking: ~2–3 min
- Multi-split stacking: ~5–8 min

In [ ]:
comparison = {}

# 1. Non-text baseline
print(f"[1/4] Non-text baseline ({N_TREES_COMPARISON} trees) ...", end=' ', flush=True)
t0 = time.time()
pipe = Pipeline([
    ('features', ColumnTransformer(nontext_transformers())),
    ('rf', RandomForestClassifier(n_estimators=N_TREES_COMPARISON, random_state=RANDOM_STATE))
])
pipe.fit(X_train, y_train)
comparison['Non-text baseline'] = roc_auc_score(y_test, pipe.predict_proba(X_test)[:, 1])
print(f"{time.time()-t0:.0f}s  AUC = {comparison['Non-text baseline']:.4f}")

# 2. Single model: all 7 columns concatenated
print(f"[2/4] Single model, all 7 cols ({N_TREES_COMPARISON} trees) ...", end=' ', flush=True)
t0 = time.time()
emb7, emb7_cols = embed_cols(text_cols, 'all7')  # cached if fwd_7 was already used
X_all7 = pd.concat([X, pd.DataFrame(emb7, columns=emb7_cols)], axis=1)
pipe = make_base_estimator(emb7_cols, n_estimators=N_TREES_COMPARISON)
pipe.fit(X_all7.loc[X_train.index], y_train)
comparison['Single model (all 7 cols)'] = roc_auc_score(
    y_test, pipe.predict_proba(X_all7.loc[X_test.index])[:, 1]
)
print(f"{time.time()-t0:.0f}s  AUC = {comparison['Single model (all 7 cols)']:.4f}")

# 3. Single-split stacking
n_trees_ss = N_TREES_COMPARISON // 2
print(f"[3/4] Single-split stacking (2 × {n_trees_ss} trees) ...", end=' ', flush=True)
t0 = time.time()
stacking_ss_final = StackingClassifier(
    estimators=[
        ('group_a', make_base_estimator(emb_a_cols, n_trees_ss)),
        ('group_b', make_base_estimator(emb_b_cols, n_trees_ss)),
    ],
    final_estimator=LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    cv=5
)
stacking_ss_final.fit(X_ss_train, y_train)
comparison['Single-split stacking'] = roc_auc_score(
    y_test, stacking_ss_final.predict_proba(X_ss_test)[:, 1]
)
print(f"{time.time()-t0:.0f}s  AUC = {comparison['Single-split stacking']:.4f}")

# 4. Multi-split stacking
n_trees_ms = N_TREES_COMPARISON // (N_SPLITS * 2)
print(f"[4/4] Multi-split stacking ({N_SPLITS*2} × {n_trees_ms} trees) ...", end=' ', flush=True)
t0 = time.time()
base_estimators_final = [
    (f'split{i+1}{g}', make_base_estimator(s[f'emb_{g}_cols'], n_trees_ms))
    for i, s in enumerate(splits)
    for g in ('a', 'b')
]
stacking_ms_final = StackingClassifier(
    estimators=base_estimators_final,
    final_estimator=LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    cv=5
)
stacking_ms_final.fit(X_ms_train, y_train)
comparison['Multi-split stacking'] = roc_auc_score(
    y_test, stacking_ms_final.predict_proba(X_ms_test)[:, 1]
)
print(f"{time.time()-t0:.0f}s  AUC = {comparison['Multi-split stacking']:.4f}")

In [ ]:
print(f"{'Configuration':<35} {'Test AUC':>9} {'Δ baseline':>11}")
print("-" * 58)
baseline = comparison['Non-text baseline']
for name, auc in comparison.items():
    delta = f"{auc - baseline:+.4f}" if name != 'Non-text baseline' else '—'
    print(f"{name:<35} {auc:>9.4f} {delta:>11}")

Both stacking configurations outperform the concatenated single model at equal total modeling
capacity. The single-split ensemble already gains from processing column groups independently,
even with a fixed grouping and no additional diversity. The multi-split ensemble improves
further: with multiple random groupings, the meta-learner leverages variance in which columns
each base estimator sees — a form of ensemble diversity unavailable to either the single-model
or single-split approaches.

This illustrates how TabuLLM's compositional design enables sophisticated ensemble architectures.
The same three building blocks — `TextColumnTransformer`, `GMMFeatureExtractor`, and sklearn's
`StackingClassifier` — support both the basic workflow in notebook 01 and the advanced patterns
here. The same framework extends further: base estimators built on different embedding models
(e.g., one using an OpenAI model and another using a Bedrock model) can be combined in the
same stack, a pattern applied in a related clinical setting to achieve further gains.